In [28]:
import polars as pl

df = pl.scan_parquet(["../../data/interim/weather/weather.parquet"])
df.filter(pl.col("Date") >= "2022-01-01").collect()

Date,Temperature,Precipitation,SnowDepth,WindSpeed,HOD
str,f64,f64,f64,f64,i64
"""2022-01-01""",0.9,0.0,0.0,null,0
"""2022-01-01""",0.3,0.0,0.0,null,1
"""2022-01-01""",0.2,0.2,0.0,null,2
"""2022-01-01""",0.6,0.1,0.0,null,3
"""2022-01-01""",0.4,0.0,0.0,null,4
…,…,…,…,…,…
"""2025-06-30""",20.3,0.0,0.0,null,18
"""2025-06-30""",14.4,0.0,0.0,null,19
"""2025-06-30""",12.4,0.0,0.0,null,20


In [29]:
na_results = {}
for col in df.collect_schema().names():
    count_df = df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'Date': 0, 'Temperature': 0, 'Precipitation': 0, 'SnowDepth': 0, 'WindSpeed': 38807, 'HOD': 0}


In [30]:
wind_df = pl.scan_parquet(["../../data/interim/weather/wind_data.parquet"])
wind_df.tail().collect()

Date,hod,Wind Speed(m/s)
str,i64,f64
"""6/30/2025""",19,2.1
"""6/30/2025""",20,3.8
"""6/30/2025""",21,2.4
"""6/30/2025""",22,0.6
"""6/30/2025""",23,0.0


In [31]:
import polars as pl

df = df.with_columns(
    pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d", strict=False),
)

wind_df = (
    wind_df.rename({
        "hod": "HOD",
        "Wind Speed(m/s)": "WindSpeed_new",
    })
    .with_columns(
        pl.col("Date").str.strptime(pl.Date, "%m/%d/%Y", strict=False),
    )
)

# 2️⃣ Right join on Date + HOD
joined_lf = df.join(
    wind_df,
    on=["Date", "HOD"],
    how="right"
)

# 3️⃣ Replace old WindSpeed with new WindSpeed_new (prefer wind data when available)
joined_lf = joined_lf.with_columns(
    pl.coalesce(pl.col("WindSpeed_new"), pl.col("WindSpeed")).alias("WindSpeed")
).drop("WindSpeed_new")

joined_lf.tail().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
14.4,0.0,0.0,2.1,2025-06-30,19
12.4,0.0,0.0,3.8,2025-06-30,20
10.5,0.0,0.0,2.4,2025-06-30,21
9.3,0.0,0.0,0.6,2025-06-30,22
null,null,null,0.0,2025-06-30,23


In [32]:
joined_lf.head().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
0.9,0.0,0.0,5.1,2022-01-01,0
0.3,0.0,0.0,3.4,2022-01-01,1
0.2,0.2,0.0,3.2,2022-01-01,2
0.6,0.1,0.0,4.9,2022-01-01,3
0.4,0.0,0.0,4.9,2022-01-01,4


In [33]:
joined_lf.sink_parquet("../../data/interim/weather/final_weather.parquet")  # streaming-safe
merged_df = pl.scan_parquet(["../../data/interim/weather/final_weather.parquet"])
merged_df.head().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
0.3,0.0,0.0,3.4,2022-01-01,1
-0.1,0.0,0.0,5.7,2022-01-01,5
0.0,1.3,2.848,3.4,2022-01-02,2
1.4,0.0,1.648,3.0,2022-01-02,6
3.6,0.0,0.0,3.1,2022-01-03,6


In [34]:
merged_df.sort("Date", "HOD").head().collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
0.9,0.0,0.0,5.1,2022-01-01,0
0.3,0.0,0.0,3.4,2022-01-01,1
0.2,0.2,0.0,3.2,2022-01-01,2
0.6,0.1,0.0,4.9,2022-01-01,3
0.4,0.0,0.0,4.9,2022-01-01,4


In [35]:
na_results = {}
for col in merged_df.collect_schema().names():
    count_df = merged_df.select(pl.col(col).is_null().sum().alias("na_count")).collect()
    na_results[col] = count_df["na_count"][0]

print(na_results)

{'Temperature': 539, 'Precipitation': 539, 'SnowDepth': 539, 'WindSpeed': 0, 'Date': 0, 'HOD': 0}


In [36]:
merged_df.sort("Date", "HOD").collect()

Temperature,Precipitation,SnowDepth,WindSpeed,Date,HOD
f64,f64,f64,f64,date,i64
0.9,0.0,0.0,5.1,2022-01-01,0
0.3,0.0,0.0,3.4,2022-01-01,1
0.2,0.2,0.0,3.2,2022-01-01,2
0.6,0.1,0.0,4.9,2022-01-01,3
0.4,0.0,0.0,4.9,2022-01-01,4
…,…,…,…,…,…
14.4,0.0,0.0,2.1,2025-06-30,19
12.4,0.0,0.0,3.8,2025-06-30,20
10.5,0.0,0.0,2.4,2025-06-30,21
